# Guesty → RC ForecastNOLA sheet

Takes the two Guesty exports (check-out + check-in) plus your current, edited
Google Sheet (downloaded as CSV), and writes a **new export** = your existing
sheet rows (untouched) **+ any reservations not already in it**.

## Each run
1. Drop the latest Guesty **check-out** and **check-in** CSVs **into the `inputs/` folder**.
2. Download your current Google Sheet as CSV **into the `inputs/` folder** too.
3. **Run all cells.**
4. Import the generated **`forecast_export.csv`** (created in this main folder) back into the Google Sheet.

**You do not need to delete old files** — the notebook always uses the *newest*
file of each type, so leftover exports are ignored. Clearing `inputs/` between
runs just keeps things tidy.

Rows are matched on **(Property, Date)** — one row per property per day. Existing
rows and your manual edits are never rewritten; new rows are appended at the
bottom. The transform rules (property normalization, exclusions, times, T/O,
city) live in `processing.py`. `property_to_city.csv` stays in this main folder
(it's config, not an input).

In [2]:
import os, glob
from datetime import datetime
import pandas as pd
from processing import (
    process_reservations, to_forecast_rows, FORECAST_COLUMNS, _canonical_key,
)

# Drop the two Guesty exports + your downloaded sheet into this folder each run:
INPUT_DIR = "inputs"
# File written for you to import back into the Google Sheet (in the main folder):
OUTPUT_CSV = "forecast_export.csv"
# Optional: force a specific downloaded-sheet filename; None = newest in inputs/:
SHEET_CSV = None


def newest_with_columns(required):
    """Newest CSV in INPUT_DIR whose header contains all `required` columns."""
    best, best_mtime = None, -1.0
    for path in glob.glob(os.path.join(INPUT_DIR, "*.csv")):
        try:
            cols = [c.strip() for c in pd.read_csv(path, nrows=0).columns]
        except Exception:
            continue
        if all(c in cols for c in required):
            m = os.path.getmtime(path)
            if m > best_mtime:
                best, best_mtime = path, m
    return best


os.makedirs(INPUT_DIR, exist_ok=True)
CHECKOUT_CSV = newest_with_columns(["CHECK-OUT DATE"])
CHECKIN_CSV = newest_with_columns(["CHECK-IN DATE"])
if SHEET_CSV is None:
    SHEET_CSV = newest_with_columns(["Property", "Date", "T/O"])

print("Input folder  :", os.path.abspath(INPUT_DIR))
print("Check-out CSV :", CHECKOUT_CSV)
print("Check-in  CSV :", CHECKIN_CSV)
print("Current sheet :", SHEET_CSV or "(none found -> output will contain all rows)")
assert CHECKOUT_CSV and CHECKIN_CSV, f"Put the Guesty check-out & check-in CSVs in ./{INPUT_DIR}/"

ImportError: cannot import name 'to_forecast_rows' from 'processing' (c:\Users\cmara\OneDrive\Documents\python\py_practice\RC_guesty_gsheets\processing.py)

In [ ]:
# 1) Transform the two Guesty exports (normalize, exclude, times, T/O, city, ...)
co = pd.read_csv(CHECKOUT_CSV); co.columns = co.columns.str.strip()
ci = pd.read_csv(CHECKIN_CSV);  ci.columns = ci.columns.str.strip()

city_seed = {}
if os.path.exists("property_to_city.csv"):
    ref = pd.read_csv("property_to_city.csv", dtype=str).fillna("")
    for _, r in ref.iterrows():
        if r["City"].strip():
            city_seed[r["Property"].strip()] = r["City"].strip()

pipeline = process_reservations(co, ci, city_seed)

# Optional owner lookup for the 'Client / Owner' column. Create property_to_owner.csv
# with columns Property,Owner to populate it; otherwise the column stays blank.
owner_by_key = {}
if os.path.exists("property_to_owner.csv"):
    own = pd.read_csv("property_to_owner.csv", dtype=str).fillna("")
    for _, r in own.iterrows():
        if r.get("Owner", "").strip():
            owner_by_key[_canonical_key(r["Property"])] = r["Owner"].strip()

today = datetime.now().strftime("%Y-%m-%d")
candidates = to_forecast_rows(pipeline, last_updated=today, owner_by_key=owner_by_key)

# 2) Load the current (edited) sheet -- the source of truth for what already exists
if SHEET_CSV and os.path.exists(SHEET_CSV):
    sheet = pd.read_csv(SHEET_CSV, dtype=str).fillna("")
else:
    sheet = pd.DataFrame(columns=FORECAST_COLUMNS)

# Match new rows' Date display format to the sheet (e.g. add ' 00:00:00' if used)
date_key = lambda v: str(v).strip()[:10]
if len(sheet) and sheet["Date"].astype(str).str.contains(r"\d\d:\d\d").any():
    candidates["Date"] = candidates["Date"].map(lambda v: f"{date_key(v)} 00:00:00")

# 3) Keep only rows whose (Property, Date) isn't already in the sheet
existing = set()
if len(sheet) and {"Property", "Date"} <= set(sheet.columns):
    existing = {(str(p).strip(), date_key(d)) for p, d in zip(sheet["Property"], sheet["Date"])}
is_new = [(str(p).strip(), date_key(d)) not in existing
          for p, d in zip(candidates["Property"], candidates["Date"])]
new_rows = candidates[is_new].reset_index(drop=True)

# 4) Append new rows beneath the existing sheet, aligned to the sheet's columns
target_cols = list(sheet.columns) if len(sheet.columns) else FORECAST_COLUMNS
new_aligned = new_rows.reindex(columns=target_cols).fillna("")
if len(sheet):
    result = pd.concat([sheet.reindex(columns=target_cols), new_aligned], ignore_index=True)
else:
    result = new_aligned
result.to_csv(OUTPUT_CSV, index=False)

print(f"Existing sheet rows : {len(sheet)}")
print(f"New rows appended   : {len(new_rows)}")
print(f"Total in {OUTPUT_CSV} : {len(result)}")

Existing sheet rows : 3626
New rows appended   : 1105
Total in forecast_export.csv : 4731


In [ ]:
# Preview the new rows being added
if len(new_rows):
    print(new_rows.to_string(index=False))
else:
    print("No new rows -- the sheet is already up to date.")

Day       Date Client / Owner                       Property Check-out Time Check-in Time   T/O Day of Week Assigned Check-out Check-in Last Updated
    2026-07-15                                  1304 Baronne       11:00 AM               FALSE   Wednesday                               2026-07-15
    2026-07-15                                  1324 Baronne       11:00 AM               FALSE   Wednesday                               2026-07-15
    2026-07-15                                  2013 Orleans       11:00 AM               FALSE   Wednesday                               2026-07-15
    2026-07-15                                   500 Jackson       11:00 AM               FALSE   Wednesday                               2026-07-15
    2026-07-16                            1163 Webberville B       11:00 AM               FALSE    Thursday                               2026-07-15
    2026-07-16                                  1316 Baronne                     04:00 PM FALSE    Thursda